# P04 — Workflow Completo: IA Generativa × FE

**Objetivo:** Pipeline completo de ponta a ponta, desde dados brutos até relatório.

**Stack:** pandas + scikit-learn + seaborn + matplotlib

**Cenário:** Investigar relação entre uso de IA generativa e funções executivas em crianças, controlando por covariáveis.

In [ ]:
# Setup
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler

import scipy.stats as stats

np.random.seed(42)
sns.set_style('whitegrid')
print("✅ Setup completo")

In [ ]:
# 1. Gerar dados sintéticos do P04
print("=== Gerando dados sintéticos ===\n")

n = 200
dados = pd.DataFrame({
    'participante': [f'P{i:03d}' for i in range(1, n+1)],
    'idade': np.random.choice([7, 8, 9], n, p=[0.3, 0.5, 0.2]),
    'sexo': np.random.choice(['F', 'M'], n),
    'ses': np.random.normal(0, 1, n),  # socioeconômico (padronizado)
})

# Variáveis preditoras
dados['uso_ia'] = np.clip(np.random.normal(3, 1.5, n), 0, 7)  # dias/semana
dados['letramento_digital'] = np.random.normal(0, 1, n)

# Mediação: uso_ia → engajamento → FE
dados['engajamento'] = 0.4 * (dados['uso_ia'] - dados['uso_ia'].mean()) / dados['uso_ia'].std() + np.random.normal(0, 0.7, n)

# Desfechos: 3 medidas de FE
dados['stroop'] = (0.3 * dados['uso_ia'] +
                  0.3 * dados['engajamento'] +
                  0.1 * dados['letramento_digital'] +
                  np.random.normal(0, 0.5, n))

dados['backward_digit'] = (0.25 * dados['uso_ia'] +
                            0.3 * dados['engajamento'] +
                            np.random.normal(0, 0.5, n))

dados['dccs'] = (0.2 * dados['uso_ia'] +
                 0.25 * dados['engajamento'] +
                 np.random.normal(0, 0.5, n))

# Composto de FE (média z-scored)
for col in ['stroop', 'backward_digit', 'dccs']:
    dados[col] = (dados[col] - dados[col].mean()) / dados[col].std()

dados['fe_composto'] = dados[['stroop', 'backward_digit', 'dccs']].mean(axis=1)

print(f"✅ Dados criados: {dados.shape[0]} participantes, {dados.shape[1]} variáveis")
dados.head()

In [ ]:
# 2. Análise exploratória (EDA)
print("\n=== Análise Exploratória ===\n")

# Estatísticas descritivas
desc = dados[['idade', 'ses', 'uso_ia', 'letramento_digital', 'engajamento', 'fe_composto']].describe()
print(desc.round(2))

# Distribuição por sexo
print("\nDistribuição por sexo:")
print(dados.groupby('sexo')[['uso_ia', 'fe_composto', 'engajamento']].mean().round(2))

In [ ]:
# 3. Heatmap de correlações
print("\n=== Heatmap de correlações ===")

vars_corr = ['idade', 'ses', 'uso_ia', 'letramento_digital', 'engajamento',
             'stroop', 'backward_digit', 'dccs', 'fe_composto']
cor_matrix = dados[vars_corr].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cor_matrix, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True,
            cbar_kws={'label': 'Correlação de Pearson'}, ax=ax)
ax.set_title('Correlações entre variáveis do P04', fontsize=14)
plt.tight_layout()
plt.savefig('20_corr_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()
print("✅ Salvo: 20_corr_heatmap.png")

In [ ]:
# 4. Scatter plots chave
print("\n=== Scatter plots ===")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, predictor in zip(axes, ['uso_ia', 'engajamento', 'letramento_digital']):
    ax.scatter(dados[predictor], dados['fe_composto'], alpha=0.5, s=20)
    
    # Linha de regressão
    slope, intercept, r, p, se = stats.linregress(dados[predictor], dados['fe_composto'])
    x_line = np.linspace(dados[predictor].min(), dados[predictor].max(), 100)
    ax.plot(x_line, slope * x_line + intercept, 'r-', linewidth=2,
            label=f'r = {r:.3f}, p = {p:.4f}')
    
    ax.set_xlabel(predictor)
    ax.set_ylabel('FE Composto (z)')
    ax.set_title(f'FE vs. {predictor}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('21_scatter_fe.png', dpi=100, bbox_inches='tight')
plt.show()
print("✅ Salvo: 21_scatter_fe.png")

In [ ]:
# 5. Análise de mediação (estilo Baron & Kenny + bootstrap)
print("\n=== Análise de Mediação ===\n")

# Padronizar
scaler = StandardScaler()
X = scaler.fit_transform(dados[['uso_ia']]).flatten()
M = scaler.fit_transform(dados[['engajamento']]).flatten()
Y = dados['fe_composto'].values

# Passo 1: X → Y (efeito total, c)
from sklearn.linear_model import LinearRegression
model_c = LinearRegression().fit(X.reshape(-1, 1), Y)
c = model_c.coef_[0]
print(f"Passo 1 — Efeito total (c): {c:.4f}")

# Passo 2: X → M (caminho a)
model_a = LinearRegression().fit(X.reshape(-1, 1), M)
a = model_a.coef_[0]
print(f"Passo 2 — Caminho a (X → M): {a:.4f}")

# Passo 3: X, M → Y (caminho b e c')
XM = np.column_stack([X, M])
model_b = LinearRegression().fit(XM, Y)
c_prime = model_b.coef_[0]  # direto
b = model_b.coef_[1]  # mediação
print(f"Passo 3 — Caminho b (M → Y): {b:.4f}")
print(f"           Efeito direto (c'): {c_prime:.4f}")

# Efeito indireto: a * b
indireto = a * b
proporcao_med = indireto / c * 100 if c != 0 else 0
print(f"\nEfeito indireto (a*b): {indireto:.4f}")
print(f"Proporção mediada: {proporcao_med:.1f}%")

# Bootstrap para IC do efeito indireto
n_boot = 5000
indireto_boot = np.zeros(n_boot)
for i in range(n_boot):
    idx = np.random.choice(len(X), len(X), replace=True)
    X_b, M_b, Y_b = X[idx], M[idx], Y[idx]
    a_b = LinearRegression().fit(X_b.reshape(-1, 1), M_b).coef_[0]
    XM_b = np.column_stack([X_b, M_b])
    b_b = LinearRegression().fit(XM_b, Y_b).coef_[1]
    indireto_boot[i] = a_b * b_b

ic_inf, ic_sup = np.percentile(indireto_boot, [2.5, 97.5])
print(f"IC 95% (bootstrap): [{ic_inf:.4f}, {ic_sup:.4f}]")

In [ ]:
# 6. Visualização do modelo de mediação
print("\n=== Diagrama de mediação ===")

fig, ax = plt.subplots(figsize=(10, 6))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

# Caixas (boxes)
from matplotlib.patches import FancyBboxPatch

def add_box(ax, x, y, w, h, text, color='lightblue'):
    box = FancyBboxPatch((x - w/2, y - h/2), w, h,
                         boxstyle="round,pad=0.1",
                         facecolor=color, edgecolor='black', linewidth=1.5)
    ax.add_patch(box)
    ax.text(x, y, text, ha='center', va='center', fontsize=12, fontweight='bold')

add_box(ax, 2, 5, 2.2, 1, 'Uso de IA\n(X)', 'lightyellow')
add_box(ax, 5, 5, 2.2, 1, 'Engajamento\n(M)', 'lightgreen')
add_box(ax, 8, 5, 2.2, 1, 'FE\n(Y)', 'lightcoral')

# Setas
ax.annotate('', xy=(4, 5), xytext=(3, 5),
            arrowprops=dict(arrowstyle='->', lw=2))
ax.text(3.5, 5.4, f'a={a:.2f}', ha='center', fontsize=11, fontweight='bold')

ax.annotate('', xy=(7, 5), xytext=(6, 5),
            arrowprops=dict(arrowstyle='->', lw=2))
ax.text(6.5, 5.4, f'b={b:.2f}', ha='center', fontsize=11, fontweight='bold')

ax.annotate('', xy=(7, 4), xytext=(3, 4.5),
            arrowprops=dict(arrowstyle='->', lw=2, linestyle='dashed'))
ax.text(5, 3.7, f"c'={c_prime:.2f}", ha='center', fontsize=11, fontweight='bold')

# Resultado
ax.text(5, 1.5, f"Efeito indireto (a×b) = {indireto:.3f}\n"
                 f"IC 95% = [{ic_inf:.3f}, {ic_sup:.3f}]\n"
                 f"Proporção mediada: {proporcao_med:.1f}%",
        ha='center', va='center', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

ax.set_title('Modelo de Mediação: Uso de IA → Engajamento → FE', fontsize=14, pad=20)
plt.tight_layout()
plt.savefig('22_mediation_diagram.png', dpi=100, bbox_inches='tight')
plt.show()
print("✅ Salvo: 22_mediation_diagram.png")

In [ ]:
# 7. Machine Learning: Random Forest para identificar preditores importantes
print("\n=== Random Forest: Importância de preditores ===\n")

X_features = dados[['idade', 'sexo', 'ses', 'uso_ia', 'letramento_digital', 'engajamento']]
X_features = pd.get_dummies(X_features, columns=['sexo'], drop_first=True)
y = dados['fe_composto']

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_features, y, test_size=0.2, random_state=42
)

# Random Forest
rf = RandomForestRegressor(n_estimators=500, max_depth=5, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# Avaliar
y_pred_train = rf.predict(X_train)
y_pred_test = rf.predict(X_test)
r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))

print(f"R² treino: {r2_train:.3f}")
print(f"R² teste:  {r2_test:.3f}")
print(f"RMSE teste: {rmse_test:.3f}")

# Importância de preditores
importancias = pd.DataFrame({
    'preditor': X_features.columns,
    'importancia': rf.feature_importances_
}).sort_values('importancia', ascending=False)

print("\nImportância de preditores:")
print(importancias.round(4).to_string(index=False))

In [ ]:
# 8. Plot de importância
print("\n=== Plot de importância ===")

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(importancias['preditor'], importancias['importancia'], color='steelblue')
ax.set_xlabel('Importância (redução de impureza)')
ax.set_title('Importância de preditores — Random Forest')
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('23_feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()
print("✅ Salvo: 23_feature_importance.png")

In [ ]:
# 9. Resumo final
print("\n" + "=" * 60)
print("RESUMO FINAL — P04")
print("=" * 60)
print(f"\nAmostra: N = {len(dados)}")
print(f"\nCorrelações principais (com fe_composto):")
for var in ['uso_ia', 'engajamento', 'letramento_digital']:
    r, p = stats.pearsonr(dados[var], dados['fe_composto'])
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    print(f"  {var:25s}: r = {r:+.3f}, p = {p:.4f} {sig}")

print(f"\nMediação:")
print(f"  Efeito total (c):      {c:+.3f}")
print(f"  Efeito direto (c'):    {c_prime:+.3f}")
print(f"  Efeito indireto (a*b): {indireto:+.3f} [{ic_inf:+.3f}, {ic_sup:+.3f}]")
print(f"  Proporção mediada:     {proporcao_med:.1f}%")

print(f"\nMachine Learning:")
print(f"  R² teste: {r2_test:.3f}")
print(f"  Preditor mais importante: {importancias.iloc[0]['preditor']}")

print("\n✅ Análise completa!")
print("\nPróximos passos:")
print("  - Substituir dados sintéticos pelos dados REAIS do P04")
print("  - SEM com lavaan (R) para análise confirmatória")
print("  - Submissão a Computers in Human Behavior")

# Conclusões

## Workflow reproduzível

Este notebook segue o padrão de **reprodutibilidade em ciência de dados**:

1. **Seed fixo** (`np.random.seed(42)`) — resultados reproduzíveis
2. **Pipeline claro** — Setup → Dados → EDA → Modelagem → Visualização
3. **Versões documentadas** — imports explícitos
4. **Outputs salvos** — PNGs + CSVs

## Quando usar cada técnica

| Técnica | Quando usar | Limitação |
|---|---|---|
| **Correlação de Pearson** | Associação linear simples | Assume normalidade, linearidade |
| **Regressão linear** | Predição + identificar preditores | Assume linearidade, homocedasticidade |
| **Mediação (Baron & Kenny)** | Testar mecanismo causal | Requer pressupostos fortes |
| **Bootstrap** | ICs sem assumir distribuição | Computacionalmente caro |
| **Random Forest** | Identificar preditores importantes | Menos interpretável que linear |

## Recursos\n
- [scikit-learn: Random Forest](https://scikit-learn.org/stable/modules/ensemble.html#forest)
- [Hayes (2022). Introduction to Mediation, Moderation, and Conditional Process Analysis](https://www.guilford.com/books/Introduction-to-Mediation-Moderation-and-Conditional-Process-Analysis/Andrew-Hayes/9781462549030)
- [VanderWeele (2015). Explanation in Causal Inference](https://global.oup.com/academic/product/explanation-in-causal-inference-9780199325870)